In [5]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [7]:
## Load trained model, scaler, and encoders

model = load_model('model.keras')

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

with open('label_encoder.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('one_hot_encoder.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)


In [8]:
# Example input data

input_data = {
    
    'CreditScore': 600,

    'Geography': 'France',

    'Gender': 'Male',

    'Age': 40,

    'Tenure': 3,

    'Balance': 60000,

    'NumOfProducts': 2,

    'HasCrCard': 1,

    'IsActiveMember': 1,

    'EstimatedSalary': 50000

}

In [ ]:
# Encode and preprocess input
input_df = pd.DataFrame([input_data])

input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

geo_encoded = onehot_encoder_geo.transform(input_df[['Geography']]).toarray()
geo_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

input_df = pd.concat([input_df.drop('Geography', axis=1), geo_df], axis=1)

input_scaled = scaler.transform(input_df[scaler.feature_names_in_])

# Predict
prediction = model.predict(input_scaled)
probability = float(prediction[0][0])

print(f'Churn Probability: {probability:.4f}')
if probability > 0.5:
    print('Prediction: Customer is likely to churn.')
else:
    print('Prediction: Customer is not likely to churn.')
